# Inference LLM on notebooks

In [ ]:
!pip -q install pyngrok

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import torch

In [3]:
from google.colab import userdata
from huggingface_hub import login

hf_token=userdata.get('HF_TOKEN')
login(token=hf_token)

In [4]:
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTHTOKEN')

In [ ]:
model_name = "quangne/text2diagram-AceMath-1.5B-Instruct-merged-geometry3k8-8-1-1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_cuda = torch.cuda.is_available()
load_dtype = (
    torch.bfloat16 if (use_cuda and torch.cuda.is_bf16_supported())
    else (torch.float16 if use_cuda else torch.float32)
 )

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=load_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
 )
model.eval()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/435 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

In [6]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
def prepare_inference_context(model):
    model.eval()
    use_cuda = torch.cuda.is_available() and str(model.device).startswith("cuda")
    compute_dtype = torch.bfloat16 if (use_cuda and torch.cuda.is_bf16_supported()) else torch.float16
    return use_cuda, compute_dtype

In [ ]:
from pyngrok import ngrok

app = FastAPI()


class Request(BaseModel):
    prompt: str
    max_new_tokens: int = 256


def generate_dsl(
    prompt_text: str,
    max_new_tokens: int = 256,
    use_cuda: bool = False,
    compute_dtype: torch.dtype = torch.float16,
 ) -> str:
    messages = [{"role": "user", "content": prompt_text}]
    if tokenizer.chat_template:
        rendered_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        rendered_prompt = f"User:\n{prompt_text}\n\nAssistant:\n"

    inputs = tokenizer(rendered_prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        if use_cuda:
            with torch.autocast(device_type="cuda", dtype=compute_dtype):
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    repetition_penalty=1.08,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.pad_token_id,
                    use_cache=True,
                )
        else:
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.08,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                use_cache=True,
            )

    prompt_len = inputs["input_ids"].shape[-1]
    generated = outputs[0][prompt_len:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


@app.post("/generate")
def generate(req: Request):
    prompt = str(req.prompt or "").strip()
    if not prompt:
        return {"response": ""}

    use_cuda, compute_dtype = prepare_inference_context(model)
    text = generate_dsl(
        prompt_text=prompt,
        max_new_tokens=req.max_new_tokens,
        use_cuda=use_cuda,
        compute_dtype=compute_dtype,
    )
    print(text)
    return {"response": text}


ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print("PUBLIC URL:", public_url)
print("GENERATE URL:", f"{public_url}/generate")

server = uvicorn.Server(
    uvicorn.Config(app, host="0.0.0.0", port=8000)
 )

await server.serve()

INFO:     Started server process [4707]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:8000 (Press CTRL+C to quit)


PUBLIC URL: NgrokTunnel: "https://raphael-remotest-windedly.ngrok-free.dev" -> "http://localhost:8000"
Cho hình chữ nhật ABCD, điểm P ∈ AB, điểm Q ∈ BC, đường thẳng PQ cắt hình chữ nhật
(rectangle (A B C D))
(define P point (segment A B))
(define Q point (segment B C))
(segment P Q)
INFO:     ::1:57306 - "POST /generate HTTP/1.1" 200 OK
Cho tam giác ABC với góc ABC = 90°, O là tâm đường tròn ngoại tiếp tam giác ABC, N là điểm thuộc đoạn thẳng AB, AC là đường kính của đường tròn ngoại tiếp
(triangle (A B C) (right B))
(define O point (circumcenter A B C))
(circle O (circumcircle A B C))
(define N point (segment A B))
(diameter A C O)
(segment A C)
(segment A B)
(segment B C)
INFO:     ::1:43408 - "POST /generate HTTP/1.1" 200 OK
Xét tam giác NOP và T là trực tâm. Các đường vuông góc với NO tại O, vuông góc với NP tại P cắt nhau ở Q
(triangle (N O P))
(define T point (orthocenter N O P))
(segment N O)
(segment N P)
(define Q point (inter-ll (segment O (define R point (segment N O))) (seg

bài đã test ok:

0. Xét hình chữ nhật JKLM. Vẽ KQ ⊥ JL (Q ∈ JL). Gọi V, T lần lượt là trung điểm của JQ và ML; R, X lần lượt là trung điểm của JK và RL. a) Chứng minh RL = TK và VX = 1/2 RL. b) Tính số đo góc 𝐵𝑀𝐾 ̂.

1. Cho hình thang VWXY (VW // XY), P là trung điểm của XY. Gọi B là giao điểm của VP và WY, N là giao điểm của WP và VX. Chứng minh BN // VW.

2. Cho tam giác ABC với góc ABC = 90°, điểm D là trung điểm của đoạn thẳng AB, điểm E là trung điểm của đoạn thẳng BC, đường thẳng DE và AC song song với DE. Tính độ dài đoạn thẳng DE nếu biết rằng AB = 6 và BC = 8.

3. Cho tam giác ABC vuông tại A, đường cao AH. Gọi I, K lần lượt là hình chiếu của H lên AB, AC.
a) Chứng minh ΔAKI ∽ ΔABC.
b) Tính diện tích tam giác ABC.
c) Tính diện tích của tứ giác AKHI.


4. Xét đường tròn (X; X), điểm Q nằm ngoài (X). Vẽ tiếp tuyến QR, QT. Kẻ đường kính TU.
a) Chứng minh Q, R, X, T thuộc một đường tròn.
b) Chứng minh XQ ⊥ RT và RU // XQ.
c) QU cắt (X) tại V, QX cắt RT tại W. Chứng minh QU.QV = QW.QX.


5. Cho ΔABC cân tại R, đường cao RU. Kẻ UW vuông góc RS tại W; UX vuông góc RT tại X. 1. Chứng minh: SU = TU. 2. Chứng minh: ΔAMN cân. 3. Gọi Y là giao điểm của WU với RT, Z là giao điểm của XU với RS, V là trung điểm của YZ. Chứng minh ba điểm X; U; V thẳng hàng.

6. Cho hình chữ nhật ABCD, điểm P ∈ AB, điểm Q ∈ BC, đường thẳng PQ cắt hình chữ nhật. Tính độ dài đoạn thẳng PQ.

7. Cho tam giác ABC với góc ABC = 90°, O là tâm đường tròn ngoại tiếp tam giác ABC, N là điểm thuộc đoạn thẳng AB, AC là đường kính của đường tròn ngoại tiếp. Tính độ dài đoạn thẳng ON.

8. 	Cho hình thang ABCD với AB ∥ CD và BC = AD. Điểm P nằm trên đoạn thẳng AB, điểm Q nằm trên đoạn thẳng BC, và đường thẳng PQ. Tính độ dài đoạn thẳng PQ nếu biết rằng chiều cao của hình thang là h.

9. Xét ΔABC , các đường cao TV , UW cắt nhau tại X .Gọi Y là trung điểm của TU .
a) Chứng minh ΔADB ∽ ΔAEC
b) Chứng minh XW.XU = XV.XT
